In [ ]:
import tensorflow as tf
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

# GPU がある場合、TensorFlow が必要な分だけメモリを使うように設定する。
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    DEVICE_NAME = "/GPU:0"
else:
    DEVICE_NAME = "/CPU:0"

print("Using device:", DEVICE_NAME)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# 再現性を少し高めるため、乱数 seed を固定する。
tf.random.set_seed(42)
np.random.seed(42)

# Colab の現在ディレクトリに train_data_2000.txt と test_data_200.txt をアップロードしてから実行する。
TRAIN_TXT = Path("train_data_2000.txt")
TEST_TXT = Path("test_data_200.txt")

# 学習とモデルの基本設定。
SEQ_LENGTH = 100
STEP = 1
BATCH_SIZE = 128
EMBEDDING_DIM = 256
LSTM_UNITS = 512
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
EPOCHS = 10
UNK_TOKEN = "<UNK>"

In [ ]:
# アップロード済みのプレーンテキストを読み込む。
if not TRAIN_TXT.exists():
    raise FileNotFoundError("train_data_2000.txt が見つかりません。Colab にファイルをアップロードしてください。")
if not TEST_TXT.exists():
    raise FileNotFoundError("test_data_200.txt が見つかりません。Colab にファイルをアップロードしてください。")

train_text = TRAIN_TXT.read_text(encoding="utf-8")
test_text = TEST_TXT.read_text(encoding="utf-8")

# 行数と文字数を確認する。今回は train 2000 行、test 200 行を想定する。
print("train lines:", len(train_text.splitlines()))
print("test lines:", len(test_text.splitlines()))
print("train chars:", len(train_text))
print("test chars:", len(test_text))
print("train preview:")
print(train_text[:500])

In [ ]:
# 学習データだけから文字辞書を作成する。
chars = sorted(set(train_text))
index_to_char = {0: UNK_TOKEN}
index_to_char.update({i + 1: ch for i, ch in enumerate(chars)})
char_to_index = {ch: i for i, ch in index_to_char.items()}

VOCAB_SIZE = len(char_to_index)
UNK_INDEX = char_to_index[UNK_TOKEN]

# 後で確認・再利用できるように、辞書も Colab 上に保存しておく。
Path("char_to_index_colab.json").write_text(
    json.dumps(char_to_index, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
Path("index_to_char_colab.json").write_text(
    json.dumps(index_to_char, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("vocab_size:", VOCAB_SIZE)
print("sample chars:", "".join(index_to_char[i] for i in range(1, min(31, VOCAB_SIZE))))

In [ ]:
def text_to_indices(text, char_to_index):
    """文字列を整数 ID の NumPy 配列に変換する。未知文字は <UNK> にする。"""
    return np.array([char_to_index.get(ch, UNK_INDEX) for ch in text], dtype=np.int32)


train_ids = text_to_indices(train_text, char_to_index)
test_ids = text_to_indices(test_text, char_to_index)

# 前100文字を入力、次の1文字を正解とするための系列数を計算する。
num_train_sequences = max(0, len(train_ids) - SEQ_LENGTH)
num_test_sequences = max(0, len(test_ids) - SEQ_LENGTH)

# Colab のメモリを守るため、shuffle buffer は最大 100,000 にする。
SHUFFLE_BUFFER_SIZE = min(num_train_sequences, 100_000)

print("train_ids:", train_ids.shape, train_ids.dtype)
print("test_ids:", test_ids.shape, test_ids.dtype)
print("num_train_sequences:", num_train_sequences)
print("num_test_sequences:", num_test_sequences)
print("shuffle_buffer_size:", SHUFFLE_BUFFER_SIZE)

In [ ]:
def make_char_dataset(encoded_ids, batch_size, shuffle=False):
    """tf.data の window を使い、メモリ効率よく滑動窓データセットを作成する。"""
    dataset = tf.data.Dataset.from_tensor_slices(encoded_ids)
    dataset = dataset.window(SEQ_LENGTH + 1, shift=STEP, drop_remainder=True)
    dataset = dataset.flat_map(lambda window: window.batch(SEQ_LENGTH + 1))
    dataset = dataset.map(
        lambda seq: (seq[:-1], seq[-1]),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=SHUFFLE_BUFFER_SIZE,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


# 学習データは shuffle し、評価データは順序を保つ。
train_dataset = make_char_dataset(train_ids, BATCH_SIZE, shuffle=True)
test_dataset = make_char_dataset(test_ids, BATCH_SIZE, shuffle=False)

# 1 batch だけ取り出して形状を確認する。
for batch_x, batch_y in train_dataset.take(1):
    print("batch_x:", batch_x.shape, batch_x.dtype)
    print("batch_y:", batch_y.shape, batch_y.dtype)

In [ ]:
# Character-Level Language Model を構築する。
# 入力: (batch, 100) の整数 ID 系列。
# 出力: (batch, vocab_size) の次文字確率分布。
with tf.device(DEVICE_NAME):
    model = keras.Sequential(
        [
            layers.Embedding(
                input_dim=VOCAB_SIZE,
                output_dim=EMBEDDING_DIM,
                name="embedding",
            ),
            layers.LSTM(
                LSTM_UNITS,
                return_sequences=False,
                name="lstm",
            ),
            layers.Dropout(DROPOUT_RATE, name="dropout"),
            layers.Dense(VOCAB_SIZE, activation="softmax", name="output"),
        ],
        name="char_lstm_language_model",
    )

    model.build(input_shape=(None, SEQ_LENGTH))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

model.summary()

In [ ]:
# 検証 loss が最も小さいモデルを保存する。
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_loss",
    save_best_only=True,
)

# 検証 loss が3エポック連続で改善しない場合、学習を早期終了する。
early_stopping_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
)

# GPU が使える場合は Colab が自動的に GPU で計算する。
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint_callback, early_stopping_callback],
)

# 学習履歴を JSON として保存しておく。
Path("history_colab.json").write_text(
    json.dumps(history.history, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

In [ ]:
# matplotlib の文字化けを避けるため、図中の文字は英語だけを使う。
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_loss, marker="o", label="Train Loss")
plt.plot(epochs_range, val_loss, marker="o", label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 学習時に保存した最良モデルを読み込む。
best_model = keras.models.load_model("best_model.keras")


def encode_seed(seed_text, char_to_index):
    """種子文字列を文字 ID に変換する。未知文字は <UNK> にする。"""
    return [char_to_index.get(ch, UNK_INDEX) for ch in seed_text]


def make_model_input(context_indices):
    """モデル入力が常に長さ100になるように、左側 padding または末尾切り出しを行う。"""
    if len(context_indices) < SEQ_LENGTH:
        context_indices = [UNK_INDEX] * (SEQ_LENGTH - len(context_indices)) + context_indices
    else:
        context_indices = context_indices[-SEQ_LENGTH:]

    return np.array([context_indices], dtype=np.int32)


def sample_next_char(model, context_indices, temperature=1.0):
    """temperature 付きで次の文字 ID をサンプリングする。"""
    if temperature <= 0:
        raise ValueError("temperature は 0 より大きい値にしてください。")

    model_input = make_model_input(context_indices)
    probabilities = model(model_input, training=False)[0]

    # モデル出力は softmax 確率なので、log を取って tf.random.categorical に渡す。
    logits = tf.math.log(probabilities + 1e-8) / temperature
    sampled_index = tf.random.categorical(tf.expand_dims(logits, axis=0), num_samples=1)[0, 0]
    return int(sampled_index.numpy())


def generate_text(model, seed_text, num_generate=200, temperature=1.0):
    """種子文字列から始めて、1文字ずつ文章を生成する。"""
    context_indices = encode_seed(seed_text, char_to_index)
    generated_chars = [seed_text]

    for _ in range(num_generate):
        next_index = sample_next_char(model, context_indices, temperature=temperature)
        context_indices.append(next_index)

        next_char = index_to_char.get(next_index, "")
        if next_char == UNK_TOKEN:
            next_char = " "
        generated_chars.append(next_char)

    return "".join(generated_chars)

In [ ]:
# 同じ seed から temperature だけを変えて、生成結果を比較する。
seed_text = "中国经济"
temperatures = [0.5, 1.0, 1.5]

for temp in temperatures:
    print("=" * 80)
    print(f"Temperature: {temp}")
    print(generate_text(best_model, seed_text, num_generate=200, temperature=temp))

In [ ]:
# 追加実験: 必要に応じて、軽量モデルも同じデータで学習して比較する。
# 時間を節約したい場合、この cell 以降は実行しなくてもよい。
LSTM_UNITS_EXP2 = 128

with tf.device(DEVICE_NAME):
    model_exp2 = keras.Sequential(
        [
            layers.Embedding(
                input_dim=VOCAB_SIZE,
                output_dim=EMBEDDING_DIM,
                name="embedding",
            ),
            layers.LSTM(
                LSTM_UNITS_EXP2,
                return_sequences=False,
                name="lstm",
            ),
            layers.Dropout(DROPOUT_RATE, name="dropout"),
            layers.Dense(VOCAB_SIZE, activation="softmax", name="output"),
        ],
        name="char_lstm_language_model_exp2",
    )

    model_exp2.build(input_shape=(None, SEQ_LENGTH))
    model_exp2.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

model_exp2.summary()

In [ ]:
# 軽量モデルの最良 checkpoint と EarlyStopping を設定する。
checkpoint_callback_exp2 = keras.callbacks.ModelCheckpoint(
    filepath="best_model_exp2.keras",
    monitor="val_loss",
    save_best_only=True,
)

early_stopping_callback_exp2 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
)

history_exp2 = model_exp2.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint_callback_exp2, early_stopping_callback_exp2],
)

Path("history_exp2_colab.json").write_text(
    json.dumps(history_exp2.history, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

In [ ]:
# Baseline と軽量モデルの validation loss を比較する。
baseline_val_loss = history.history["val_loss"]
exp2_val_loss = history_exp2.history["val_loss"]

baseline_epochs = range(1, len(baseline_val_loss) + 1)
exp2_epochs = range(1, len(exp2_val_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(baseline_epochs, baseline_val_loss, marker="o", label="Baseline LSTM 512")
plt.plot(exp2_epochs, exp2_val_loss, marker="o", label="Light LSTM 128")
plt.title("Validation Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 2つの保存済みモデルを読み込み、同じ seed と temperature で生成結果を比較する。
best_model_exp1 = keras.models.load_model("best_model.keras")
best_model_exp2 = keras.models.load_model("best_model_exp2.keras")

for temp in temperatures:
    print("=" * 80)
    print(f"Temperature: {temp}")
    print("\n[Experiment 1: Baseline LSTM 512]")
    print(generate_text(best_model_exp1, seed_text, num_generate=200, temperature=temp))
    print("\n[Experiment 2: Light LSTM 128]")
    print(generate_text(best_model_exp2, seed_text, num_generate=200, temperature=temp))
    print()

In [ ]:
# 実験結果の簡単な分析を表示する。
analysis_text = """
簡単な分析:
LSTM のユニット数を 512 から 128 に減らすと、モデルのパラメータ数が大きく減るため、1エポックあたりの学習時間は短くなる。一方で、文字列の長い依存関係や語彙の使い分けを表現する能力も小さくなるため、validation loss は Baseline より下がりにくくなる可能性が高い。

生成文については、512 units のモデルの方が中国語ニュースらしい語句や句読点の流れを比較的保ちやすいと期待される。128 units のモデルは、低い temperature では同じような文字や語句を繰り返しやすく、高い temperature では文脈から外れた文字が増えやすい。したがって、生成品質を見るときは loss だけでなく、temperature ごとの実際の文章の自然さも合わせて評価する必要がある。
"""

print(analysis_text)